<a href="https://colab.research.google.com/github/rist-kobe/HPC-Programming/blob/main/Tuning/sample_code/sample_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Scalar Programming - Sample Code

After running Setup, each section can be executed independently.


## Setup


Install GNU Fortran. The NVIDIA HPC SDK is optional and commented out.


In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y build-essential gfortran curl gnupg

# Optional: install NVIDIA HPC SDK (large download).
# Uncomment only when needed.
#!curl -fsSL https://developer.download.nvidia.com/hpc-sdk/ubuntu/DEB-GPG-KEY-NVIDIA-HPC-SDK | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg
#!echo 'deb [signed-by=/usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg] https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64 /' | sudo tee /etc/apt/sources.list.d/nvhpc.list
#!sudo apt-get update -y
#!sudo apt-get install -y nvhpc-22-7-cuda-multi


Display information about the execution environment (compilers, kernel, CPU).


In [ ]:
!gcc --version
!g++ --version
!gfortran --version
!uname -a
!lscpu


Clone the repository and move to the working directory.


In [ ]:
import os

%cd /content
!rm -rf HPC-Programming
!git clone https://github.com/rist-kobe/HPC-Programming.git

os.environ["SAMPLE_CODE_DIR"] = "/content/HPC-Programming/Tuning/sample_code"
%cd {os.environ["SAMPLE_CODE_DIR"]}
!ls


## 01_timer: Hand-coded timers and gprof


In [ ]:
import os
%cd {os.environ["SAMPLE_CODE_DIR"]}/01_timer


Following `01_timer/README.md`, the cells below build and run the same C example three times, selecting the timing method with the `MODE` variable (`elp`, `cpu`, `gprof`).

The README covers four language variants — C (`src/c`/`tests/c`), Fortran (`src/fortran`/`tests/fortran`), Fortran with C timer via ISO_C_BINDING (`src/fortran_c`/`tests/fortran_c`), and C++ with `std::chrono` (`src/cpp`/`tests/cpp`). This notebook automates the **C version** only.
> **Note:** You can run this notebook with the other language variants by replacing `c` with `fortran`, `fortran_c`, or `cpp` in the paths used in the code cells (e.g., `src/c` → `src/fortran` and `tests/c` → `tests/fortran`).

Before running, inspect `src/c/main.c` and note the differences between the two timed loops:

- routine 1 calls `sub1` 100000 times
- routine 2 calls `sub2` 200000 times
- `sub1` calls `sub3` twice per call, while `sub2` calls `sub3` four times per call; `sub3` evaluates `sin()` once each time it is called

The timing output is labeled `sub1:`/`sub2:` in the program output.

In [ ]:
%%bash
sed -n "43,118p" src/c/main.c


**Step 1: Measure elapsed (wall-clock) time.**

Following the README, the manual steps for the C version are:
```
$ cd src/c
$ make MODE=elp
$ cd ../../tests/c
$ bash run.sh MODE=elp
```
The notebook automates this by rebuilding with `make veryclean && make MODE=elp` in `src/c` and then running `bash run.sh MODE=elp` in `tests/c`.
Check `outfile` for two labeled lines:
```
sub1: Elapsed time (sec) = ...
sub2: Elapsed time (sec) = ...
```

In [ ]:
%%bash
cd src/c
make veryclean && make MODE=elp
cd ../../tests/c
bash run.sh MODE=elp
echo '--- outfile ---'
tail -n +1 outfile


Compare the elapsed times of the two timed loops (routine 1 calling `sub1`, and routine 2 calling `sub2`) and consider which one is more expensive and why.

**Step 2: Measure CPU time.**

Following the README, the manual steps for the C version are:
```bash
$ cd src/c
$ make veryclean && make MODE=cpu
$ cd ../../tests/c
$ bash run.sh MODE=cpu
```

Check `outfile` for:
```
sub1: CPU time (sec)     = ...
sub2: CPU time (sec)     = ...
```

In [ ]:
%%bash
cd src/c
make veryclean && make MODE=cpu
cd ../../tests/c
bash run.sh MODE=cpu
echo '--- outfile ---'
tail -n +1 outfile


Compare CPU time with the elapsed time from Step 1.
For a single-threaded program (like this sample), CPU time is at most the
elapsed time; a noticeable gap indicates the process was waiting (I/O,
other processes, etc.) rather than computing. For a multi-threaded program
(e.g., OpenMP), the CPU timer used here (`get_cpu_time()`, based on
`CLOCK_PROCESS_CPUTIME_ID`) sums
the CPU time of all threads, so CPU time can exceed the elapsed time —
the ratio CPU time / elapsed time is a rough measure of how many cores
were kept busy.
Note that the resolution of the CPU timer is coarser; you may have to
enlarge the array size (`nn` in `main.c`) or the loop counts to obtain
meaningful values.

**Step 3: Profile with `gprof`.**

Following the README, the manual steps for the C version are:
```
$ cd src/c
$ make veryclean && make MODE=gprof
$ cd ../../tests/c
$ bash run.sh MODE=gprof
```
When you run `bash run.sh MODE=gprof` (with a `-pg` build), the script generates `prof.out` automatically.

In this mode, `outfile` contains the program output (`a[0] = ...` lines), `gmon.out` contains the raw profiling data, and `prof.out` contains the `gprof` report. Examine the flat profile and call graph in `prof.out` to identify hotspots.

To clean up after profiling:
```
$ cd tests/c
$ rm -f gmon.out prof.out outfile
```

In [ ]:
%%bash
cd src/c
make veryclean && make MODE=gprof
cd ../../tests/c
bash run.sh MODE=gprof
echo '--- outfile ---'
tail -n +1 outfile
echo '--- profiling files ---'
for f in gmon.out prof.out; do
  if [[ -f $f ]]; then ls -l "$f"; else echo "$f not found (was the build done with MODE=gprof?)"; fi
done
echo '--- prof.out (first 60 lines) ---'
head -n 60 prof.out 2>/dev/null || echo 'prof.out not found'


Examine the flat profile and the call graph in `prof.out` to find the functions corresponding to the hotspot, and confirm that the result is consistent with the hand-coded timer measurements.

**Questions to consider:**
1. Which function is the hotspot, and how do the call counts of `sub1`, `sub2`, and `sub3` explain it?
2. When do elapsed time and CPU time differ, and which one should you use for tuning?
3. What are the pros and cons of hand-coded timers vs. `gprof`?
4. How do timer implementations differ across C, Fortran, C++, and Fortran-C interoperability?
5. What are the trade-offs between POSIX timers (`clock_gettime`), Fortran intrinsics (`system_clock`, `cpu_time`), and C++ `std::chrono`?
6. For your use case, which language and timer approach is most appropriate and why?

## 02_timer-res: Check the resolution of the timer


In [ ]:
import os
%cd {os.environ["SAMPLE_CODE_DIR"]}/02_timer-res


Build and run following the C example in the README.


Compile: run `make` in `src/c`.


In [ ]:
%%bash
cd src/c
make


Run: execute `tests/c/run.sh`.


In [ ]:
%%bash
cd tests/c
bash run.sh


## 03_prof-ex: Exercise on performance analysis with `gprof` (and `perf`)

This exercise can be partially reproduced in Google Colab, but there are important limitations.
- `gprof` may work in Colab if the program is compiled with profiling enabled and profiling output can be generated normally.
- `perf` is generally **not available in Colab** because it depends on Linux kernel performance counters, which are typically restricted in managed virtualized environments.
- Therefore, when using Colab, treat this section mainly as a **`gprof` exercise**.
- If you want to try `perf`, run the same example on a local Ubuntu system or on a Linux server where you have appropriate permissions.


In [ ]:
import os
%cd {os.environ["SAMPLE_CODE_DIR"]}/03_prof-ex


As stated in the README prerequisites, this sample requires external Mersenne Twister source files that are **not included** in this repository, so copy them into `src/c` (and/or `src/fortran`) before running `make`.


Before building:
- place `mt19937ar.c` and `mt19937ar.h` in `src/c` for the C version;
- place the required Fortran Mersenne Twister source in `src/fortran` for the Fortran version, if needed.


In [ ]:
%%bash
cd src/c
if [[ -f mt19937ar.c && -f mt19937ar.h ]]; then
  echo "Mersenne Twister sources found."
else
  echo "Missing mt19937ar.c / mt19937ar.h. Download mt19937ar.sep.tgz (see ../../README.md in 03_prof-ex/), extract it in src/c, then re-run."
  # wget https://www.math.sci.hiroshima-u.ac.jp/m-mat/MT/MT2002/CODES/mt19937ar.sep.tgz
  # tar xzvf mt19937ar.sep.tgz
fi


Compile: run `make` when the required files are present.


In [ ]:
%%bash
cd src/c
if [[ -f mt19937ar.c && -f mt19937ar.h ]]; then
  make
else
  echo "Skip compile until mt19937ar sources are added."
fi


Run + gprof: `tests/c/run.sh`. Use `tests/perf` as needed.


In [ ]:
%%bash
cd tests/c
if [[ -x ../../src/c/diffuse.x ]]; then
  bash run.sh
else
  echo "Skip run because ../../src/c/diffuse.x is not built yet."
fi
echo "perf examples are under tests/perf"


## 04_alloc2d: Check address of 2D arrays


In [ ]:
import os
%cd {os.environ["SAMPLE_CODE_DIR"]}/04_alloc2d


Build and run following the C example in the README and check `outlist`.


Compile + Run


In [ ]:
%%bash
cd src/c
make
cd ../../tests/c
bash run.sh
echo "--- outlist ---"
tail -n +1 outlist


## 05_mattp: Loop blocking: transpose a matrix


In [ ]:
import os
%cd {os.environ["SAMPLE_CODE_DIR"]}/05_mattp


Build and run following the C example in the README and check `outlist`.


In [ ]:
%%bash
cd src/c
make
cd ../../tests/c
bash run.sh
echo "--- outlist ---"
tail -n +1 outlist


## 06_matmatp: Blocking in matrix-matrix products on dense matrices


In [ ]:
import os
%cd {os.environ["SAMPLE_CODE_DIR"]}/06_matmatp


Following the C example in the README, build in `src/c` and then run in `tests/c`.


In [ ]:
%%bash
cd src/c
make
cd ../../tests/c
bash run.sh
echo "--- generated outputs ---"
ls -1 output.* 2>/dev/null || true


## 07_stripmining: Strip mining


In [ ]:
import os
%cd {os.environ["SAMPLE_CODE_DIR"]}/07_stripmining


Build and run following the C example in the README (note: the README says this technique is usually not recommended).


In [ ]:
%%bash
cd src/c
make
cd ../../tests/c
bash run.sh


## 08_thrashing: Thrashing


In [ ]:
import os
%cd {os.environ["SAMPLE_CODE_DIR"]}/08_thrashing


Build and run following the C example in the README and check `output` and `add.info`.


In [ ]:
%%bash
cd src/c
make
cd ../../tests/c
bash run.sh
echo "--- output (tail) ---"
tail -n 20 output
echo "--- add.info (tail) ---"
tail -n 20 add.info


## 09_unroll: Loop unrolling: on outer loop


In [ ]:
import os
%cd {os.environ["SAMPLE_CODE_DIR"]}/09_unroll


Build and run following the C example in the README and check `outlist`.


In [ ]:
%%bash
cd src/c
make
cd ../../tests/c
bash run.sh
echo "--- outlist ---"
tail -n +1 outlist


## 10_simd-add: Example of SIMD: Vector addition


In [ ]:
import os
%cd {os.environ["SAMPLE_CODE_DIR"]}/10_simd-add


Use `make all` as in the GNU example in the README, then run `run.sh` and check the logs.


In [ ]:
%%bash
cd src/c
make all
cd ../../tests/c
bash run.sh
echo "--- run_v.log (tail) ---"
tail -n 20 run_v.log
echo "--- run_nv.log (tail) ---"
tail -n 20 run_nv.log


## 11_simd-nsimple: Example of SIMD: Non-simple loops


In [ ]:
import os
%cd {os.environ["SAMPLE_CODE_DIR"]}/11_simd-nsimple


Run `make all` and `run.sh` as in the GNU example in the README.


In [ ]:
%%bash
cd src/c
make all
cd ../../tests/c
bash run.sh
echo "--- run_v.log (tail) ---"
tail -n 20 run_v.log
echo "--- run_nv.log (tail) ---"
tail -n 20 run_nv.log


## 12_swp: Example of software pipelining


In [ ]:
import os
%cd {os.environ["SAMPLE_CODE_DIR"]}/12_swp


Build and run following the C example in the README and check `outfile.*`.


In [ ]:
%%bash
cd src/c
make
cd ../../tests/c
bash run.sh
echo "--- generated outputs ---"
ls -1 outfile.* 2>/dev/null || true


## 13_polynomial: Comparison between different implementations of a compute-bound kernel, n-th order polynomial


In [ ]:
import os
%cd {os.environ["SAMPLE_CODE_DIR"]}/13_polynomial


Following the README, compile and run in all subdirectories so the different implementations can be compared.


In [ ]:
%%bash
for d in c cpp cpp.et14 cpp.etp cpp.oo cpp.valarray f90 f90.forall f08.concurrent; do
  echo "=== build src/$d ==="
  (cd src/$d && make) || { echo "[WARN] build failed in src/$d"; continue; }
  echo "=== run tests/$d/run.sh ==="
  (cd tests/$d && bash run.sh) || echo "[WARN] run failed in tests/$d"
done
echo "--- sample logs ---"
for d in c cpp cpp.oo f90; do
  echo "[$d]"
  tail -n 5 tests/$d/run.log || true
done


## 14_endian: Check endianness


In [ ]:
import os
%cd {os.environ["SAMPLE_CODE_DIR"]}/14_endian


Build and run following the C example in the README and check `logfile`.


In [ ]:
%%bash
cd src/c
make
cd ../../tests/c
bash run.sh
echo "--- logfile ---"
tail -n +1 logfile
echo "--- lscpu endian check ---"
lscpu | grep -i -E "endian|byte" || true


## 15_io-format: Formatted (text) vs. binary outputs


In [ ]:
import os
%cd {os.environ["SAMPLE_CODE_DIR"]}/15_io-format


Build and run following the C example in the README and check `logfile`.


In [ ]:
%%bash
cd src/c
make
cd ../../tests/c
bash run.sh
echo "--- logfile ---"
tail -n +1 logfile
